# Negotiate, Then Solve: training the constraint compiler

LoRA fine-tuning of **Qwen3.5-4B** (or 2B) to turn timetable requests into the
same `ParseOutput` JSON as the Gemini System Two parser, then export to GGUF for
local inference on a 6 GB laptop GPU.

**Before running**

1. On your machine: `uv run python -m nts.compiler` writes `data/compiler/`
   (`train.jsonl`, `val.jsonl`, `test.jsonl`). Upload that folder as a Kaggle
   dataset (e.g. `nts-compiler`) and add it to this notebook (*Add Input*).
2. *Settings*: **Accelerator: GPU T4** (x1 or x2; only one is used), **Internet: on**.
3. Run with `TRIAL = True` first (20 steps, a few minutes) to check that
   installation, fp16 on the T4 and memory are fine. Then set `TRIAL = False`
   and *Run all*.

**Why 16-bit LoRA, not QLoRA:** Unsloth advises against 4-bit (QLoRA) training
for Qwen3.5. A 4B model in 16-bit needs about 10 GB, which fits a T4 (16 GB).
The T4 has no bf16, so training runs in fp16.

The test split is not used here. Score the exported model locally with the
project's evaluation so all parsers are compared on the same requests.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Unsloth trains on one GPU

from unsloth import FastLanguageModel, is_bfloat16_supported  # import before transformers
import torch, transformers, trl

print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
assert int(transformers.__version__.split(".")[0]) >= 5, (
    "Qwen3.5 needs transformers v5. Restart the kernel (Run > Restart) and run from this cell.")

## Settings

In [ ]:
import glob
from pathlib import Path

MODEL = "Qwen/Qwen3.5-4B"   # "Qwen/Qwen3.5-2B" for the small comparison
TRIAL = True                # True: 20 steps + 10 val requests, no export
EPOCHS = 3
LR = 2e-4
MAX_LEN = 1280              # prompts are about 800 tokens, answers under 200
SEED = 3407

DATA = Path(sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))[0]).parent
OUT = Path("/kaggle/working") / (MODEL.split("/")[-1].lower() + "-nts")
print("data:", DATA, "| output:", OUT)

## Model and LoRA adapters

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_LEN,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_LEN,
)
# Qwen3.5 is multimodal: the loader may return a processor; its text tokenizer
# is what the chat template and trainer need.
tok = getattr(tokenizer, "tokenizer", tokenizer)
print("eos:", repr(tok.eos_token))
print(f"GPU memory after load: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

## Data

The prompt is rendered with the chat template and **thinking disabled**, exactly
as the local server will render it, and loss is computed on the JSON answer only.

In [ ]:
import json
from datasets import load_dataset

def render_prompt(messages):
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)

def to_pc(row):
    return {"prompt": render_prompt(row["messages"]),
            "completion": row["messages"][2]["content"] + tok.eos_token}

raw = load_dataset("json", data_files={"train": str(DATA / "train.jsonl"), "val": str(DATA / "val.jsonl")})
ds = raw.map(to_pc, remove_columns=raw["train"].column_names)
print(ds)
print(ds["train"][0]["prompt"][-400:])
print(ds["train"][0]["completion"])
n_tokens = [len(tok(p + c)["input_ids"]) for p, c in zip(ds["train"]["prompt"], ds["train"]["completion"])]
print("tokens per example: max", max(n_tokens), "mean", sum(n_tokens) // len(n_tokens))
assert max(n_tokens) <= MAX_LEN, "raise MAX_LEN"

## Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tok,
    train_dataset = ds["train"],
    eval_dataset = ds["val"],
    args = SFTConfig(
        max_seq_length = MAX_LEN,
        completion_only_loss = True,          # learn the JSON, not the prompt
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,      # effective batch 16
        num_train_epochs = EPOCHS,
        max_steps = 20 if TRIAL else -1,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        weight_decay = 0.01,
        optim = "adamw_8bit",
        fp16 = not is_bfloat16_supported(),   # T4: fp16
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        eval_strategy = "no" if TRIAL else "epoch",
        save_strategy = "no",
        report_to = "none",
        seed = SEED,
        output_dir = str(OUT / "checkpoints"),
    ),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} min, "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

## Quick check on the validation split

Greedy decoding on validation requests: is the output valid JSON, and is it
identical to the gold answer? This is only a sanity check; the real scores come
from the project's evaluation (tiers, authority, semantic match).

In [ ]:
FastLanguageModel.for_inference(model)
val_rows = [json.loads(l) for l in open(DATA / "val.jsonl", encoding="utf-8")]
if TRIAL:
    val_rows = val_rows[:10]

valid = exact = 0
misses = []
for row in val_rows:
    inputs = tok(render_prompt(row["messages"]), return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = json.loads(row["messages"][2]["content"])
    try:
        pred = json.loads(text)
        valid += 1
        exact += pred == gold
        if pred != gold:
            misses.append((row["id"], text, gold))
    except json.JSONDecodeError:
        misses.append((row["id"], text, gold))

n = len(val_rows)
print(f"valid JSON {valid}/{n}, identical to gold {exact}/{n}")
for rid, text, gold in misses[:8]:
    print("---", rid, "\npred:", text[:300], "\ngold:", json.dumps(gold)[:300])

## Export

* the LoRA adapter (small; keep it to re-export later);
* GGUF files for llama.cpp / Ollama: **Q5_K_M** (recommended on the 6 GB GPU,
  about 3.1 GB for 4B) and Q4_K_M (about 2.7 GB).

Unsloth merges the adapter into the 16-bit weights before quantising and builds
llama.cpp the first time, which takes several minutes.

In [ ]:
if TRIAL:
    print("TRIAL run: set TRIAL = False and run all to train fully and export.")
else:
    model.save_pretrained(str(OUT / "lora"))
    tok.save_pretrained(str(OUT / "lora"))
    model.save_pretrained_gguf(str(OUT / "gguf"), tokenizer, quantization_method=["q5_k_m", "q4_k_m"])
    for f in sorted(Path("/kaggle/working").rglob("*.gguf")):
        print(f, f"{f.stat().st_size / 2**30:.2f} GB")

## Next: run it locally

Download the `.gguf` file from the notebook's *Output* tab, then on the laptop:

```sh
llama-server -m qwen3.5-4b-nts.Q5_K_M.gguf --jinja -ngl 99 -c 4096 --port 8080
```

Thinking must stay off, as in training: send `"chat_template_kwargs": {"enable_thinking": false}`
with each request (the project's local client will do this). Keep the system prompt
and directory format from `nts.compiler` unchanged, since the model was trained on them.